In [9]:
import pandas as pd
from sklearn.linear_model import LinearRegression

# 1. Dosyayı oku
veriler_path = 'tahminsonuclari.csv'  # kendi dosya yolunu ver
veriler = pd.read_csv(veriler_path)

# 2. Kullanılacak değişkenler
hedef_degiskenler = [
    "nufus", "hastane", "yatak",
    "uzman", "asistan", "pratisyen", 
    "diger", "hemsire", "ebe", "disci", "eczaci"
]

# 3. Şehirler listesi
sehirler = veriler['il_adı'].unique()

# 4. 2025 tahminlerini tutacak boş liste
tahmin_listesi = []

# 5. Her şehir için 2025 tahmini
for sehir in sehirler:
    sehir_veri = veriler[veriler['il_adı'] == sehir]
    if sehir_veri.empty:
        continue
    
    tahmin = {'il_adı': sehir, 'yıl': 2025}
    
    for degisken in hedef_degiskenler:
        X = sehir_veri[['yıl']]
        y = sehir_veri[degisken]
        
        model = LinearRegression()
        model.fit(X, y)
        
        tahmin_degeri = model.predict(pd.DataFrame({'yıl': [2025]}))[0]
        tahmin[degisken] = max(tahmin_degeri, 0)
    
    tahmin_listesi.append(tahmin)

# 6. 2025 verisini oluştur
veriler_2025 = pd.DataFrame(tahmin_listesi)

# 7. Eski veriler ve yeni 2025 verilerini birleştir
tum_yillar = pd.concat([veriler, veriler_2025], ignore_index=True)

# 8. Personel/Yatak ve Personel/Hastane oranlarını hesapla
personel_turleri = [
    "uzman", "asistan", "pratisyen", 
    "diger", "hemsire", "ebe", "disci", "eczaci"
]

for personel in personel_turleri:
    tum_yillar[f"{personel}_per_yatak"] = tum_yillar[personel] / tum_yillar["yatak"]
    tum_yillar[f"{personel}_per_hastane"] = tum_yillar[personel] / tum_yillar["hastane"]

# 9. Sıralamayı düzeltelim: önce şehir, sonra yıl sıralı olsun
tum_yillar = tum_yillar.sort_values(by=['il_adı', 'yıl']).reset_index(drop=True)

# 10. CSV olarak kaydet
tum_yillar.to_csv('2025verileri.csv', index=False, float_format='%.8f')

print("✅ 2002-2025 verileri şehir ve yıl sıralı şekilde kaydedildi!")


✅ 2002-2025 verileri şehir ve yıl sıralı şekilde kaydedildi!
